# DEMO TỔNG HỢP — TransUNet trên Synapse (DS200.Q21)

**Nhóm:** 23520228 - Nguyễn Hải Đăng • 23520055 - Nguyễn Bi Anh

Notebook này gom **tất cả kết quả** của đồ án vào một chỗ để **chạy và quay video demo**:

1. **Bài toán & dữ liệu** Synapse (8 cơ quan + background)
2. **Reproduce TransUNet** (giống tác giả) — `demo.ipynb`
3. **Baseline R50-U-Net** — `demo.ipynb`
4. **Ablation skip-connection** (`n_skip = 0, 1, 3`) — `ablationskip.ipynb`
5. **Cải tiến Attention Gate** (`TransUNet_AttnSkip`) — `attentiongate.ipynb`

---
## Cách dùng (2 phần)

| Phần | Cần gì | Mục đích |
|------|--------|----------|
| **A. Tổng hợp kết quả** | Không cần GPU/dữ liệu | Bảng + biểu đồ so sánh — chạy ngay, luôn ra hình |
| **B. Trực quan (overlay)** | GPU + dữ liệu + checkpoint | Chạy lại inference, vẽ overlay dự đoán |

> **Quay video nhanh:** chỉ cần chạy **Phần A** (vài giây). Muốn show dự đoán trên ảnh thật thì chạy thêm **Phần B**.

**Số liệu cuối cùng (đã train trên Colab A100):**

| Mô hình | Mean DSC (%) | Mean HD95 (mm) |
|---------|-------------|----------------|
| TransUNet (n_skip=3) | **77.65** | 31.22 |
| R50-U-Net | 76.31 | **29.59** |
| TransUNet_AttnSkip | 76.81 | 33.89 |

# PHẦN A — Tổng hợp kết quả (chạy ngay, không cần GPU)

Các số dưới đây là kết quả thật đã thu được từ `demo.ipynb`, `huong3.ipynb`, `attentiongate.ipynb`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============ KẾT QUẢ CUỐI CÙNG (đã chạy trên Colab) ============
# DSC: cao = tốt | HD95: thấp = tốt
RESULTS = {
    'R50-U-Net':            {'dsc': 76.31, 'hd95': 29.59, 'epoch': 150, 'note': 'Baseline CNN'},
    'TransUNet (n_skip=3)': {'dsc': 77.65, 'hd95': 31.22, 'epoch': 150, 'note': 'Reproduce paper'},
    'TransUNet_AttnSkip':   {'dsc': 76.81, 'hd95': 33.89, 'epoch': 150, 'note': 'Attention Gate (cai tien)'},
}
PAPER_TU = {'dsc': 77.48, 'hd95': 31.69}   # TransUNet goc (Chen et al. 2021)

print('=' * 60)
print(f"{'Mo hinh':<24}{'DSC(%)':>9}{'HD95(mm)':>11}")
print('-' * 60)
for name, r in RESULTS.items():
    print(f"{name:<24}{r['dsc']:>9.2f}{r['hd95']:>11.2f}")
print(f"{'Paper TransUNet':<24}{PAPER_TU['dsc']:>9.2f}{PAPER_TU['hd95']:>11.2f}")
print('=' * 60)

In [ ]:
# ============ BANG SO SANH (matplotlib table) ============
names = list(RESULTS.keys())
fig, ax = plt.subplots(figsize=(11, 3.2)); ax.axis('off')
cols = ['Mo hinh', 'Mean DSC (%)', 'Mean HD95 (mm)', 'Ghi chu']
rows = [[n, f"{RESULTS[n]['dsc']:.2f}", f"{RESULTS[n]['hd95']:.2f}", RESULTS[n]['note']] for n in names]
rows.append(['Paper TransUNet', f"{PAPER_TU['dsc']:.2f}", f"{PAPER_TU['hd95']:.2f}", 'Chen et al. 2021'])
tbl = ax.table(cellText=rows, colLabels=cols, loc='center', cellLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(11); tbl.scale(1.2, 1.6)
for j in range(len(cols)):
    tbl[0, j].set_facecolor('#4472C4'); tbl[0, j].set_text_props(color='white', fontweight='bold')
# To xanh: DSC tot nhat va HD95 tot nhat
best_dsc = int(np.argmax([RESULTS[n]['dsc'] for n in names]))
best_hd = int(np.argmin([RESULTS[n]['hd95'] for n in names]))
tbl[best_dsc + 1, 1].set_facecolor('#C6EFCE')
tbl[best_hd + 1, 2].set_facecolor('#C6EFCE')
ax.set_title('So sanh dinh luong cac mo hinh (Synapse, 12 test volumes)',
             fontsize=13, fontweight='bold', pad=16)
plt.tight_layout(); plt.savefig('demo_final_table.png', dpi=150, bbox_inches='tight'); plt.show()
print('Xanh = tot nhat o cot do (DSC cao nhat / HD95 thap nhat).')

In [ ]:
# ============ BIEU DO COT: DSC va HD95 ============
names = list(RESULTS.keys())
dsc = [RESULTS[n]['dsc'] for n in names]
hd95 = [RESULTS[n]['hd95'] for n in names]
short = ['U-Net', 'TransUNet\n(n_skip=3)', 'AttnSkip']
colors = ['#5B9BD5', '#70AD47', '#ED7D31']

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
b0 = ax[0].bar(short, dsc, color=colors)
ax[0].axhline(PAPER_TU['dsc'], ls='--', color='gray', label=f"Paper TU = {PAPER_TU['dsc']}%")
ax[0].set_title('Mean DSC (%) — cao hon = tot hon', fontweight='bold')
ax[0].set_ylim(70, 80); ax[0].legend()
for b, v in zip(b0, dsc): ax[0].text(b.get_x() + b.get_width() / 2, v + 0.1, f'{v:.2f}', ha='center', fontweight='bold')

b1 = ax[1].bar(short, hd95, color=colors)
ax[1].axhline(PAPER_TU['hd95'], ls='--', color='gray', label=f"Paper TU = {PAPER_TU['hd95']}")
ax[1].set_title('Mean HD95 (mm) — thap hon = tot hon', fontweight='bold')
ax[1].set_ylim(25, 36); ax[1].legend()
for b, v in zip(b1, hd95): ax[1].text(b.get_x() + b.get_width() / 2, v + 0.1, f'{v:.2f}', ha='center', fontweight='bold')

plt.suptitle('R50-U-Net vs TransUNet vs TransUNet_AttnSkip', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('demo_final_bars.png', dpi=150, bbox_inches='tight'); plt.show()

## Ablation skip-connection (`ablationskip.ipynb`)

Chứng minh vai trò skip theo **Figure 2** của paper: bỏ skip → DSC giảm mạnh.

| `n_skip` | Tên | Mean DSC | Mean HD95 |
|----------|-----|----------|-----------|
| 0 | R50-ViT-CUP | 68.94 | 32.36 |
| 1 | TransUNet (1 skip) | 73.88 | 30.71 |
| 3 | TransUNet đầy đủ | 77.65 | 31.22 |

> Lưu ý: `n_skip=0,1` train 60 epoch (deadline), `n_skip=3` train 150 epoch — nên số tuyệt đối thấp hơn paper một chút, nhưng **xu hướng 0 < 1 < 3 đúng như paper**.

In [ ]:
# ============ ABLATION n_skip ============
ABL = {
    '0 (R50-ViT-CUP)':       {'dsc': 68.94, 'hd95': 32.36, 'epoch': 60},
    '1 (TransUNet 1-skip)':  {'dsc': 73.88, 'hd95': 30.71, 'epoch': 60},
    '3 (TransUNet day du)':  {'dsc': 77.65, 'hd95': 31.22, 'epoch': 150},
}
labels = list(ABL.keys())
dsc_a = [ABL[k]['dsc'] for k in labels]
hd_a = [ABL[k]['hd95'] for k in labels]

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
ax[0].plot(['0', '1', '3'], dsc_a, 'o-', lw=2.5, ms=10, color='#70AD47')
ax[0].set_title('DSC tang theo so skip (n_skip)', fontweight='bold')
ax[0].set_xlabel('n_skip'); ax[0].set_ylabel('Mean DSC (%)'); ax[0].grid(alpha=0.3)
for x, v in zip(['0', '1', '3'], dsc_a): ax[0].text(x, v + 0.4, f'{v:.2f}', ha='center', fontweight='bold')

ax[1].bar(['0', '1', '3'], hd_a, color='#ED7D31')
ax[1].set_title('HD95 theo n_skip (thap = tot)', fontweight='bold')
ax[1].set_xlabel('n_skip'); ax[1].set_ylabel('Mean HD95 (mm)'); ax[1].set_ylim(28, 34)
for i, v in enumerate(hd_a): ax[1].text(i, v + 0.1, f'{v:.2f}', ha='center', fontweight='bold')

plt.suptitle('Ablation skip-connection (TransUNet) — Synapse', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('demo_final_ablation.png', dpi=150, bbox_inches='tight'); plt.show()
print('Ket luan: bo skip (n_skip=0) lam DSC tut ~9% -> skip-connection rat quan trong.')

# PHẦN B — Trực quan dự đoán (cần GPU + dữ liệu + checkpoint)

Phần này **chạy lại inference thật** để vẽ overlay dự đoán của các mô hình lên ảnh CT.

**Yêu cầu:**
1. Runtime → Change runtime type → **GPU**
2. Có **dữ liệu Synapse** (tải từ Kaggle như các notebook trước)
3. Có **checkpoint** đã train. Nếu bạn đã lưu checkpoint lên **Google Drive**, chỉ cần trỏ đúng đường dẫn ở cell cấu hình.

> Nếu chưa có checkpoint/dữ liệu, **bỏ qua Phần B** — Phần A đã đủ cho video. Phần B chỉ thêm hình overlay đẹp.

In [ ]:
# ============ CAU HINH PHAN B ============
RUN_SETUP   = True   # cai dat + clone repo + tai du lieu Synapse
USE_DRIVE   = True   # mount Google Drive de load checkpoint da luu

# Tai khoan Kaggle (de tai du lieu Synapse). Dien key cua ban neu can.
KAGGLE_USERNAME = 'ynzboyvt'
KAGGLE_KEY      = 'KGAT_e1129cdba4fc46f3eb3dfdfb0c7b7f9a'

# Thu muc chua checkpoint tren Drive (chinh lai cho dung cho ban).
# Notebook se tu tim file epoch_*.pth trong cac thu muc nay.
DRIVE_CKPT_ROOT = '/content/drive/MyDrive/DS200_checkpoints'

print('Cau hinh OK. RUN_SETUP =', RUN_SETUP, '| USE_DRIVE =', USE_DRIVE)

In [ ]:
# ============ SETUP: cai dat, clone repo, tai du lieu, ViT weights ============
if RUN_SETUP:
    import os, shutil
    !pip install -q ml-collections medpy SimpleITK tensorboardX h5py scipy matplotlib kaggle

    !rm -rf /content/TransUNet
    !git clone -q https://github.com/Beckschen/TransUNet.git

    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
        f.write('{"username":"%s","key":"%s"}' % (KAGGLE_USERNAME, KAGGLE_KEY))
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

    if not os.path.exists('/content/data/Synapse/test_vol_h5'):
        !kaggle datasets download -d dogcdt/synapse -p /content/synapse_zip
        !unzip -q -o /content/synapse_zip/*.zip -d /content/synapse_temp
        for folder in ['train_npz', 'test_vol_h5']:
            src = f'/content/synapse_temp/Synapse/{folder}'
            dst = f'/content/data/Synapse/{folder}'
            os.makedirs(dst, exist_ok=True)
            for fname in os.listdir(src):
                shutil.move(os.path.join(src, fname), os.path.join(dst, fname))
        !rm -rf /content/synapse_zip /content/synapse_temp

    os.makedirs('/content/model/vit_checkpoint/imagenet21k', exist_ok=True)
    if not os.path.exists('/content/model/vit_checkpoint/imagenet21k/R50+ViT-B_16.npz'):
        !wget -q https://storage.googleapis.com/vit_models/imagenet21k/R50+ViT-B_16.npz -O /content/model/vit_checkpoint/imagenet21k/R50+ViT-B_16.npz

    cfg = '/content/TransUNet/networks/vit_seg_configs.py'
    with open(cfg, 'r') as f: txt = f.read()
    txt = txt.replace("'../model/vit_checkpoint/imagenet21k/", "'/content/model/vit_checkpoint/imagenet21k/")
    with open(cfg, 'w') as f: f.write(txt)

    !touch /content/TransUNet/datasets/__init__.py /content/TransUNet/networks/__init__.py

    t = len(os.listdir('/content/data/Synapse/train_npz')) if os.path.exists('/content/data/Synapse/train_npz') else 0
    e = len(os.listdir('/content/data/Synapse/test_vol_h5')) if os.path.exists('/content/data/Synapse/test_vol_h5') else 0
    print(f'\nTrain: {t} slices | Test: {e} volumes | SETUP DONE')
else:
    print('Bo qua setup (RUN_SETUP = False).')

In [ ]:
%%writefile /content/TransUNet/unet_baseline.py
import torch, torch.nn as nn
from networks.vit_seg_modeling import Conv2dReLU, DecoderBlock, SegmentationHead
from networks.vit_seg_modeling_resnet_skip import ResNetV2

class R50UNet(nn.Module):
    def __init__(self, num_classes=9):
        super().__init__()
        self.encoder = ResNetV2(block_units=(3, 4, 9), width_factor=1)
        self.conv_more = Conv2dReLU(1024, 512, kernel_size=3, padding=1, use_batchnorm=True)
        dc = (256, 128, 64, 16)
        sc = [512, 256, 64, 0]
        ic = [512] + list(dc[:-1])
        self.blocks = nn.ModuleList([DecoderBlock(i, o, s) for i, o, s in zip(ic, dc, sc)])
        self.seg_head = SegmentationHead(in_channels=16, out_channels=num_classes, kernel_size=3)

    def forward(self, x):
        if x.size()[1] == 1: x = x.repeat(1, 3, 1, 1)
        x, features = self.encoder(x)
        x = self.conv_more(x)
        for i, blk in enumerate(self.blocks):
            x = blk(x, skip=features[i] if i < 3 else None)
        return self.seg_head(x)

In [ ]:
%%writefile /content/TransUNet/networks/vit_seg_attnskip.py
import numpy as np
import torch
import torch.nn as nn
from .vit_seg_modeling import VisionTransformer, Conv2dReLU

class AttentionGate(nn.Module):
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(nn.Conv2d(F_g, F_int, 1, 1, 0, bias=True), nn.BatchNorm2d(F_int))
        self.W_x = nn.Sequential(nn.Conv2d(F_l, F_int, 1, 1, 0, bias=True), nn.BatchNorm2d(F_int))
        self.psi = nn.Sequential(nn.Conv2d(F_int, 1, 1, 1, 0, bias=True), nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)

    def forward(self, g, x):
        g1 = self.W_g(g); x1 = self.W_x(x)
        alpha = self.psi(self.relu(g1 + x1))
        return x * alpha

class DecoderBlockAG(nn.Module):
    def __init__(self, in_channels, out_channels, skip_channels=0, use_batchnorm=True):
        super().__init__()
        self.conv1 = Conv2dReLU(in_channels + skip_channels, out_channels, 3, padding=1, use_batchnorm=use_batchnorm)
        self.conv2 = Conv2dReLU(out_channels, out_channels, 3, padding=1, use_batchnorm=use_batchnorm)
        self.up = nn.UpsamplingBilinear2d(scale_factor=2)
        self.attn_gate = AttentionGate(F_g=in_channels, F_l=skip_channels, F_int=max(skip_channels // 2, 1)) if skip_channels > 0 else None

    def forward(self, x, skip=None):
        x = self.up(x)
        if skip is not None:
            if self.attn_gate is not None:
                skip = self.attn_gate(g=x, x=skip)
            x = torch.cat([x, skip], dim=1)
        return self.conv2(self.conv1(x))

class DecoderCupAG(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        head_channels = 512
        self.conv_more = Conv2dReLU(config.hidden_size, head_channels, 3, padding=1, use_batchnorm=True)
        decoder_channels = config.decoder_channels
        in_channels = [head_channels] + list(decoder_channels[:-1])
        out_channels = decoder_channels
        if self.config.n_skip != 0:
            skip_channels = self.config.skip_channels
            for i in range(4 - self.config.n_skip):
                skip_channels[3 - i] = 0
        else:
            skip_channels = [0, 0, 0, 0]
        self.blocks = nn.ModuleList([DecoderBlockAG(ic, oc, sc) for ic, oc, sc in zip(in_channels, out_channels, skip_channels)])

    def forward(self, hidden_states, features=None):
        B, n_patch, hidden = hidden_states.size()
        h, w = int(np.sqrt(n_patch)), int(np.sqrt(n_patch))
        x = hidden_states.permute(0, 2, 1).contiguous().view(B, hidden, h, w)
        x = self.conv_more(x)
        for i, blk in enumerate(self.blocks):
            skip = features[i] if (features is not None and i < self.config.n_skip) else None
            x = blk(x, skip=skip)
        return x

class ViT_AttnSkip(VisionTransformer):
    def __init__(self, config, img_size=224, num_classes=21843, zero_head=False, vis=False):
        super().__init__(config, img_size=img_size, num_classes=num_classes, zero_head=zero_head, vis=vis)
        self.decoder = DecoderCupAG(config)

In [ ]:
# ============ MOUNT DRIVE + TIM CHECKPOINT ============
import glob, os

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

def find_ckpt(patterns):
    """Tra ve checkpoint moi nhat khop voi bat ky pattern nao."""
    for p in patterns:
        files = sorted(glob.glob(p))
        if files:
            return files[-1]
    return None

# Tim o ca /content/model (neu vua train) lan Drive (neu da luu)
CKPT = {
    'transunet': find_ckpt([
        '/content/model/TU_Synapse224/*/epoch_*.pth',
        f'{DRIVE_CKPT_ROOT}/TU_Synapse224/*/epoch_*.pth',
        f'{DRIVE_CKPT_ROOT}/**/TU_*skip3*epoch_*.pth',
    ]),
    'unet': find_ckpt([
        '/content/model/UNet_Synapse224/epoch_*.pth',
        f'{DRIVE_CKPT_ROOT}/UNet_Synapse224/epoch_*.pth',
        f'{DRIVE_CKPT_ROOT}/**/UNet*epoch_*.pth',
    ]),
    'attnskip': find_ckpt([
        '/content/model/TU_AttnSkip_Synapse224/epoch_*.pth',
        f'{DRIVE_CKPT_ROOT}/TU_AttnSkip_Synapse224/epoch_*.pth',
        f'{DRIVE_CKPT_ROOT}/**/*AttnSkip*epoch_*.pth',
    ]),
}
for k, v in CKPT.items():
    print(f"{k:<12}: {v if v else 'KHONG TIM THAY (se bo qua model nay)'}")

In [ ]:
# ============ XEM THU DU LIEU: vai lat CT + nhan that ============
import h5py, numpy as np, matplotlib.pyplot as plt

ORGAN = {1: 'Aorta', 2: 'Gallbladder', 3: 'L.Kidney', 4: 'R.Kidney',
         5: 'Liver', 6: 'Pancreas', 7: 'Spleen', 8: 'Stomach'}
CLR = np.array([[0, 0, 0], [255, 0, 0], [0, 255, 0], [0, 0, 255], [255, 255, 0],
                [255, 0, 255], [0, 255, 255], [128, 0, 255], [255, 128, 0]]) / 255.

def overlay(img, lbl, a=0.45):
    im = (img - img.min()) / (img.max() - img.min() + 1e-8)
    rgb = np.stack([im] * 3, -1).copy()
    for c in range(1, 9):
        m = lbl == c
        if m.any():
            for ch in range(3): rgb[:, :, ch][m] = (1 - a) * rgb[:, :, ch][m] + a * CLR[c][ch]
    return np.clip(rgb, 0, 1)

test_dir = '/content/data/Synapse/test_vol_h5'
import matplotlib.patches as mpatches
cases = sorted([f.replace('.npy.h5', '') for f in os.listdir(test_dir)])
data = h5py.File(os.path.join(test_dir, f'{cases[0]}.npy.h5'), 'r')
imgs, lbls = data['image'][:], data['label'][:]
best = sorted(range(imgs.shape[0]), key=lambda s: -len(np.unique(lbls[s])))[:3]
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
for j, si in enumerate(best):
    ax[j].imshow(overlay(imgs[si], lbls[si].astype(int)))
    ax[j].set_title(f'{cases[0]} - slice {si}', fontweight='bold'); ax[j].axis('off')
patches = [mpatches.Patch(color=CLR[i], label=ORGAN[i]) for i in range(1, 9)]
fig.legend(handles=patches, loc='lower center', ncol=8, fontsize=9, bbox_to_anchor=(0.5, -0.04))
fig.suptitle('Du lieu Synapse: anh CT + nhan that (ground truth)', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.savefig('demo_final_data.png', dpi=150, bbox_inches='tight'); plt.show()

In [ ]:
# ============ LOAD MODEL + VE OVERLAY SO SANH ============
import sys, torch
from scipy.ndimage import zoom
sys.path.insert(0, '/content/TransUNet')
%cd /content/TransUNet
from networks.vit_seg_modeling import VisionTransformer as ViT_seg, CONFIGS
from networks.vit_seg_attnskip import ViT_AttnSkip
from unet_baseline import R50UNet

models = {}

if CKPT['transunet']:
    cfg = CONFIGS['R50-ViT-B_16']; cfg.n_classes = 9; cfg.n_skip = 3; cfg.patches.grid = (14, 14)
    m = ViT_seg(cfg, img_size=224, num_classes=9).cuda()
    m.load_state_dict(torch.load(CKPT['transunet'])); m.eval(); models['TransUNet'] = m

if CKPT['unet']:
    m = R50UNet(num_classes=9).cuda()
    m.load_state_dict(torch.load(CKPT['unet'])); m.eval(); models['R50-U-Net'] = m

if CKPT['attnskip']:
    cfg2 = CONFIGS['R50-ViT-B_16']; cfg2.n_classes = 9; cfg2.n_skip = 3; cfg2.patches.grid = (14, 14)
    m = ViT_AttnSkip(cfg2, img_size=224, num_classes=9).cuda()
    m.load_state_dict(torch.load(CKPT['attnskip'])); m.eval(); models['TransUNet_AttnSkip'] = m

print('Da load:', list(models.keys()) if models else 'KHONG co model nao (kiem tra checkpoint).')

def pred(model, slc):
    h, w = slc.shape
    s = zoom(slc, (224 / h, 224 / w), order=3) if (h != 224 or w != 224) else slc
    with torch.no_grad():
        o = model(torch.from_numpy(s).unsqueeze(0).unsqueeze(0).float().cuda())
        o = torch.argmax(torch.softmax(o, 1), 1).squeeze(0).cpu().numpy()
    return zoom(o, (h / 224, w / 224), order=0).astype(int) if (h != 224 or w != 224) else o.astype(int)

In [ ]:
# ============ OVERLAY: CT | Ground Truth | tung model ============
if models:
    case = cases[0]
    data = h5py.File(os.path.join(test_dir, f'{case}.npy.h5'), 'r')
    imgs, lbls = data['image'][:], data['label'][:]
    best = sorted(range(imgs.shape[0]), key=lambda s: -len(np.unique(lbls[s])))[:2]
    ncol = 2 + len(models)
    for si in best:
        gt = lbls[si].astype(int)
        fig, ax = plt.subplots(1, ncol, figsize=(5.2 * ncol, 5.2))
        im_n = (imgs[si] - imgs[si].min()) / (imgs[si].max() - imgs[si].min() + 1e-8)
        ax[0].imshow(im_n, cmap='gray'); ax[0].set_title('CT Image', fontweight='bold')
        ax[1].imshow(overlay(imgs[si], gt)); ax[1].set_title('Ground Truth', fontweight='bold')
        for k, (name, mdl) in enumerate(models.items()):
            ax[2 + k].imshow(overlay(imgs[si], pred(mdl, imgs[si]))); ax[2 + k].set_title(name, fontweight='bold')
        for a_ in ax: a_.axis('off')
        patches = [mpatches.Patch(color=CLR[i], label=ORGAN[i]) for i in range(1, 9)]
        fig.legend(handles=patches, loc='lower center', ncol=8, fontsize=9, bbox_to_anchor=(0.5, -0.03))
        fig.suptitle(f'{case} - slice {si}', fontsize=14, fontweight='bold')
        plt.tight_layout(); plt.savefig(f'demo_final_overlay_{case}_s{si}.png', dpi=150, bbox_inches='tight'); plt.show()
else:
    print('Khong co model nao duoc load -> bo qua overlay. Phan A van du cho video.')

# Kết luận — nói gì trong video

1. **Bài toán:** phân đoạn 8 cơ quan trên CT bụng (Synapse), 2211 slice train / 12 volume test.
2. **Reproduce TransUNet** đạt **77.65% DSC** ≈ paper **77.48%** → tái hiện thành công.
3. **R50-U-Net** (baseline CNN): DSC thấp hơn (76.31%) nhưng **HD95 tốt hơn** (29.59) → trade-off accuracy vs biên.
4. **Ablation skip:** `n_skip = 0 → 1 → 3` cho DSC **68.94 → 73.88 → 77.65** → đúng Figure 2 paper, **skip-connection rất quan trọng**.
5. **Cải tiến Attention Gate** (`TransUNet_AttnSkip`): **76.81% / 33.89** — **không vượt** TransUNet gốc. Negative result: gate train từ đầu + khác hướng paper (Mục 4.4 là Transformer *trong* skip).

**Model tốt nhất:**
- Theo **DSC**: **TransUNet (n_skip=3)**.
- Theo **HD95 (biên)**: **R50-U-Net**.

> Tất cả hình đã được lưu ra file `demo_final_*.png` để chèn vào báo cáo/slide.